### Skipping these as I already have them in my workspace
#### Load Transactions Dataset
```
transactions_df = spark.read.csv(
    "/FileStore/tables/Transactions_500.csv",
    header=True,
    inferSchema=True
)

display(transactions_df)
```

#### Save Bronze Table
```
transactions_df.write.mode("overwrite").saveAsTable(
    "transactions_bronze"
)
```

In [0]:
# Create Customer Dimension
customer_dim = transactions_df.select(
    "customer_id",
    "country"
).dropDuplicates()

display(customer_dim)


customer_id,country
4251,UK
5393,India
3523,Australia
9434,UK
4799,Australia
9219,UK
4760,USA
5711,UK
3445,Canada
9929,Canada


In [0]:
# Create Transaction Fact Table
fact_transactions = transactions_df.select(
    "transaction_id",
    "customer_id",
    "amount",
    "transaction_date",
    "status"
)

display(fact_transactions)


transaction_id,customer_id,amount,transaction_date,status
100001,6050,11882.92,2020-12-03,Completed
100002,3216,9513.66,2024-05-05,Active
100003,9645,14640.97,2019-02-04,Completed
100004,5775,7937.4,2024-08-07,Closed
100005,4816,11986.44,2022-02-12,Closed
100006,5234,9264.21,2018-08-22,Active
100007,5560,11508.34,2018-01-10,Pending
100008,8377,6719.36,2026-01-05,Active
100009,1206,5841.95,2022-07-20,Pending
100010,4061,13881.72,2025-11-09,Pending


### Sensor Architecture Modeling

In [0]:
# Load Sensor Dataset
sensor_df = spark.read.csv(
    "/Volumes/workspace/default/Volume/sensor_data.csv",
    header=True,
    inferSchema=True
)

In [0]:
# Create Sensor Dimension
sensor_dim = sensor_df.select(
    "sensor_id",
    "machine_id"
).dropDuplicates()


In [0]:
# Create Sensor Fact Table
sensor_fact = sensor_df.select(
    "sensor_id",
    "timestamp",
    "temperature",
    "pressure",
    "vibration",
    "status"
)


### Silver Layer Architecture

In [0]:
# Create Processed Tables
fact_transactions.write.mode("overwrite").saveAsTable(
    "transactions_silver"
)

sensor_fact.write.mode("overwrite").saveAsTable(
    "sensor_silver"
)


### Gold Layer – Aggregated Reporting

In [0]:
from pyspark.sql.functions import sum

gold_sales = transactions_df.groupBy(
    "country"
).agg(
    sum("amount").alias("total_sales")
)

display(gold_sales)


country,total_sales
Australia,760928.1999999998
USA,763331.24
Canada,826041.9700000004
India,810365.9700000004
UK,696092.0799999998


In [0]:
# Save Gold Table
gold_sales.write.mode("overwrite").saveAsTable(
    "sales_gold"
)


### Delta Architecture & Versioning

In [0]:
# Save Delta Table
transactions_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("transactions_delta_arch")
